In [ ]:
# RUN ON SERAC
# SET UP SO IF WHILLANS DRIVE PLUGGED INTO THE BACK, FILE PATHS ARE CORRECT

# Zachary Katz
# zachary_katz@mines.edu
# 02 August 2024

# Plot Figure S2 for Whillans Catalog Paper
# Tide comparison for GZ05

# Note NO NODAL refers to no nodal correction applied.

# Imports
import os
import sys

sys.path.insert(
    0,
    "/Volumes/Whillans/src/Catalog",
)
sys.path.insert(
    0,
    "/Volumes/Whillans/src/Tides",
)
import Catalog
import Tides

import logging
import numpy as np
import datetime
import pyTMD

dir = "/Volumes/Whillans/csrs_2024/all"
stas = ["gz05"]
stas = ["gz20"]
# year_arr = [["2007"],["2008"],["2009"],["2010"],["2011"],["2012"],['2013'],['2014'],['2015'],["2016"],["2017"],["2018"],["2019"]]
year_arr = [["2018"]]
filename = f"{stas[0]}{year_arr[0][0]}"
# Detection parameters
max_gap_len = 120  # Maximum gap length to interpolate [seconds]

# Log Level
# Currently implemented: ERROR, WARNING, INFO, DEBUG
# Each level also includes all levels above it.
loglevel = "INFO"


def set_log_level(loglevel: str) -> None:
    """Set logging level for the run

    Parameters
    ----------
    loglevel : str
        Logging level
    """
    FORMAT = "%(asctime)s %(name)s %(levelname)s: %(message)s"
    DATEFMT = "%Y-%m-%d %H:%M:%S"
    if loglevel == "ERROR":
        logging.basicConfig(
            format=FORMAT, datefmt=DATEFMT, level=logging.ERROR, force=True
        )
    elif loglevel == "WARNING":
        logging.basicConfig(
            format=FORMAT, datefmt=DATEFMT, level=logging.WARNING, force=True
        )
    elif loglevel == "INFO":
        logging.basicConfig(
            format=FORMAT, datefmt=DATEFMT, level=logging.INFO, force=True
        )
    elif loglevel == "DEBUG":
        logging.basicConfig(
            format=FORMAT, datefmt=DATEFMT, level=logging.DEBUG, force=True
        )

In [17]:
# Setup logger
set_log_level(loglevel)
logger = logging.getLogger(__name__)

# Create Catalog
for years in year_arr:
    cats = []
    for sta in stas:
        interpolation_time, run = Catalog.set_interpolation_time(sta, years)
        if run:
            logger.info(f"Creating Dataframe for {sta}")
            cat = Catalog.Datastream(
                os.path.join(dir, sta), sta, years, interpolation_time
            )
            logger.info(f"Interpolating {sta}")
            if not cat.data.empty:
                cat.findgaps(max_gap_len)
                cats.append(cat)

2025-03-14 14:42:19 __main__ INFO: Creating Dataframe for gz20
2025-03-14 14:42:34 __main__ INFO: Interpolating gz20
2025-03-14 14:42:41 Catalog INFO: 2018-03-08 15:07:45 2018-03-08 15:09:45 0 days 00:02:00
2025-03-14 14:42:51 Catalog INFO: 2018-06-01 01:43:45 2018-06-01 01:45:45 0 days 00:02:00
2025-03-14 14:42:57 Catalog INFO: 2018-10-21 03:06:45 2018-10-21 03:08:45 0 days 00:02:00
2025-03-14 14:42:59 Catalog INFO: 2018-11-06 09:17:30 2018-11-06 09:19:15 0 days 00:01:45
2025-03-14 14:43:00 Catalog INFO: 2018-11-09 14:59:45 2018-11-09 15:00:30 0 days 00:00:45


In [ ]:
# Set where to calculate tides
lats = [-84.2986]
lons = [-164.5206]

# Set model location
model_loc = "/mnt/c/Users/ZacharyKatz/Desktop/Research/Background"
model = "CATS2008-v2023"

tide_mod = Tides.Tide(model_loc=model_loc, model=model)

# Model tides at same times
tide_times = cats[0].data["time"]
modeled_tides = tide_mod.tidal_elevation(lons, lats, tide_times)

IndexError: list index out of range

In [ ]:
# Extract tidal constants (amplitude and phase)
# Set up pytmd model
model_loc = "/mnt/c/Users/ZacharyKatz/Desktop/Research/Background"
model_name = "CATS2008-v2023"

model = pyTMD.io.model(model_loc, format="netcdf").elevation(model_name)
constituents = pyTMD.io.OTIS.read_constants(
    model.grid_file,
    model.model_file,
    model.projection,
    type=model.type,
    grid=model.format,
)
c = constituents.fields
amp, ph, D = pyTMD.io.OTIS.interpolate_constants(
    np.atleast_1d(lons),
    np.atleast_1d(lats),
    constituents,
    model.projection,
    type=model.type,
    method="spline",
    extrapolate=True,
)

['m2', 's2', 'n2', 'k2', 'k1', 'o1', 'p1', 'q1', 'mf', 'mm']
[[0.12530586123466492 0.14462734758853912 0.13967309892177582
  0.06251237541437149 0.44746169447898865 0.35937005281448364
  0.15769679844379425 0.07832222431898117 0.03063744306564331
  0.016669640317559242]]
[[220.23974029747197 159.94927436071023 125.7258465431743
  174.624714434048 199.75769686351788 183.9426124064068 196.3462086183933
  174.0377279767771 196.78493898953988 188.77273094214306]]
[11.868412441187804]


TypeError: unsupported format string passed to MaskedArray.__format__

In [ ]:
# Print constitutents, amplitudes, and phases
for i in range(len(c)):
    print(f"Constitutent: {c[i]}, Amplitude: {amp[0][i]}, Phase: {ph[0][i]}")

ref_time = datetime(2025, 1, 1, 0, 0)  # Example reference date
# Compute minor constituents
# minor_constituents = pyTMD.predict.compute_minor(major_constituents, ref_time)

Constitutent: m2, Amplitude: 0.12530586123466492, Phase: 220.23974029747197
Constitutent: s2, Amplitude: 0.14462734758853912, Phase: 159.94927436071023
Constitutent: n2, Amplitude: 0.13967309892177582, Phase: 125.7258465431743
Constitutent: k2, Amplitude: 0.06251237541437149, Phase: 174.624714434048
Constitutent: k1, Amplitude: 0.44746169447898865, Phase: 199.75769686351788
Constitutent: o1, Amplitude: 0.35937005281448364, Phase: 183.9426124064068
Constitutent: p1, Amplitude: 0.15769679844379425, Phase: 196.3462086183933
Constitutent: q1, Amplitude: 0.07832222431898117, Phase: 174.0377279767771
Constitutent: mf, Amplitude: 0.03063744306564331, Phase: 196.78493898953988
Constitutent: mm, Amplitude: 0.016669640317559242, Phase: 188.77273094214306


In [4]:
# Find difference
data = (cats[0].data["elevation"] - np.mean(cats[0].data["elevation"])) * 100
diff = [data[i] - modeled_tides.data[i] for i in range(len(data))]

NameError: name 'modeled_tides' is not defined

In [18]:
# UTIDE SOLN

from utide import solve

# Print data
data = (cats[0].data["elevation"] - np.mean(cats[0].data["elevation"])) * 100
data_m = data / 100
times = cats[0].data["time"]
print(len(times))

1375826


In [10]:
# UTide harmonic analysis of data to extract amplitudes and phase
# Just runs with 32gb memory
soln = solve(times[:], data_m[:], lat=-84.2986, method="ols", conf_int="MC")

solve: matrix prep ... solution ... done.


In [19]:
# Utide Harmonic analysis without nodal tide
soln = solve(
    times[:], data_m[:], lat=-84.2986, method="ols", conf_int="MC", nodal=False
)

solve: matrix prep ... solution ... done.


In [21]:
print(soln)

A       : [4.15991770e-01 3.34370252e-01 1.49031058e-01 1.45355585e-01
 1.28049074e-01 1.19855392e-01 6.78218001e-02 4.34495192e-02
 3.83048792e-02 3.76030184e-02 2.89112611e-02 2.69241647e-02
 2.55918142e-02 2.26045179e-02 2.12043865e-02 2.07409498e-02
 1.87469063e-02 1.70800910e-02 1.70583072e-02 1.64540529e-02
 1.35526470e-02 1.25791834e-02 1.24200258e-02 1.21092612e-02
 1.15118595e-02 1.08564573e-02 8.84799456e-03 8.54199706e-03
 8.50117932e-03 8.40618063e-03 5.61054951e-03 5.48840047e-03
 5.41630659e-03 5.40377782e-03 5.27218933e-03 3.92060131e-03
 3.59466200e-03 3.08633963e-03 3.05091394e-03 2.74029334e-03
 2.43662291e-03 2.21162331e-03 2.10160620e-03 1.80671730e-03
 1.76323467e-03 1.65845962e-03 1.47821978e-03 1.42175612e-03
 1.19304443e-03 1.18799098e-03 1.15220364e-03 1.09044602e-03
 7.60831942e-04 7.59317770e-04 7.09449137e-04 5.85783410e-04
 5.48186813e-04 5.09038906e-04 1.79871127e-04]
A_ci    : [0.00206812 0.00184084 0.00185969 0.00207222 0.00211218 0.00246955
 0.0019522  

In [22]:
for i, constitutent in enumerate(soln.name):
    print(
        f"Constitutent: {constitutent}, Amplitude: {soln.A[i]}, A_ci: {soln.A_ci[i]} Phase: {soln.g[i]}, g_ci: {soln.g_ci[i]}"
    )

# Save to txt file
# A_ci amplitude confidence interval
# g_ci phase confidence interval
with open("gz202018_NO_NODAL.txt", "w") as f:
    for i, constitutent in enumerate(soln.name):
        f.write(
            f"Constitutent: {constitutent}, Amplitude: {soln.A[i]}, A_ci: {soln.A_ci[i]} Phase: {soln.g[i]}, g_ci: {soln.g_ci[i]} \n"
        )

Constitutent: K1, Amplitude: 0.4159917699956972, A_ci: 0.0020681153513511003 Phase: 206.9010824006462, g_ci: 0.2398391028399823
Constitutent: O1, Amplitude: 0.3343702519721603, A_ci: 0.0018408416294018154 Phase: 172.07272963085575, g_ci: 0.3045252057330661
Constitutent: S2, Amplitude: 0.1490310583128777, A_ci: 0.0018596873095503411 Phase: 166.3525236982192, g_ci: 0.7952990027635256
Constitutent: P1, Amplitude: 0.14535558537936122, A_ci: 0.002072215659571481 Phase: 199.01936619839452, g_ci: 0.6992106149130459
Constitutent: N2, Amplitude: 0.1280490744155345, A_ci: 0.0021121762021675946 Phase: 142.02534970606285, g_ci: 0.8819683699095238
Constitutent: M2, Amplitude: 0.11985539227798747, A_ci: 0.0024695516172178553 Phase: 218.99931000478105, g_ci: 1.0094879991589167
Constitutent: Q1, Amplitude: 0.06782180011684569, A_ci: 0.0019522023643772774 Phase: 163.7681825471078, g_ci: 1.5845352711638048
Constitutent: MU2, Amplitude: 0.04344951918159581, A_ci: 0.0021732645639512124 Phase: 119.40364475

In [14]:
for i, constitutent in enumerate(soln.name):
    print(
        f"Constitutent: {constitutent}, Amplitude: {soln.A[i]}, A_ci: {soln.A_ci[i]} Phase: {soln.g[i]}, g_ci: {soln.g_ci[i]}"
    )

# Save to txt file
# A_ci amplitude confidence interval
# g_ci phase confidence interval
with open("gz052011.txt", "w") as f:
    for i, constitutent in enumerate(soln.name):
        f.write(
            f"Constitutent: {constitutent}, Amplitude: {soln.A[i]}, A_ci: {soln.A_ci[i]} Phase: {soln.g[i]}, g_ci: {soln.g_ci[i]} \n"
        )

Constitutent: K1, Amplitude: 0.4522723389386209, A_ci: 0.0013016041921314857 Phase: 198.6971938241173, g_ci: 0.17530596212954885
Constitutent: O1, Amplitude: 0.3708879697188209, A_ci: 0.0015938336001599992 Phase: 182.26986319936725, g_ci: 0.20852312593300215
Constitutent: P1, Amplitude: 0.14448184879587378, A_ci: 0.0015664759882467595 Phase: 195.8838036093247, g_ci: 0.6237506658570631
Constitutent: S2, Amplitude: 0.13821042458060068, A_ci: 0.001816391964702755 Phase: 164.57505610467425, g_ci: 0.6124836496588935
Constitutent: M2, Amplitude: 0.1254003872190879, A_ci: 0.0017273694649073941 Phase: 221.06157230749704, g_ci: 0.7261325427900276
Constitutent: N2, Amplitude: 0.11161405973915396, A_ci: 0.001861601044875947 Phase: 132.8224558595256, g_ci: 0.7614377696258342
Constitutent: Q1, Amplitude: 0.07644500352207097, A_ci: 0.001420444706332842 Phase: 172.69222964676902, g_ci: 1.0200674565354155
Constitutent: K2, Amplitude: 0.04820856034263214, A_ci: 0.0016896969668976924 Phase: 174.37283922

In [28]:
# Load text files and make chart
import pandas as pd

files = ["gz052011.txt", "gz052013.txt", "gz202018.txt"]
files = ["gz052011_NO_NODAL.txt", "gz052013_NO_NODAL.txt", "gz202018_NO_NODAL.txt"]
dfs = [pd.read_csv(f, header=None, sep="\s+") for f in files]

# Print table

consts = ["M2", "S2", "N2", "K2", "K1", "O1", "P1", "Q1", "MF", "MSF", "MM", "SSA"]

print(
    "Constituent\tGZ05 2011 Amplitude [m]\tGZ05 2011 Phase [°]\tGZ05 2013 Amplitude [m]\tGZ05 2013 Phase [°]\tGZ20 2018 Amplitude [m]\tGZ20 2018 Phase [°]\tCATS 2011 Amplitude [m]\tCATS 2011 Phase [°]"
)
for const in consts:
    amps = []
    a_cis = []
    phs = []
    ph_cis = []
    for df in dfs:
        row = df[df[1] == f"{const},"]
        amp = float(row[3].iloc[0][:-1])
        a_ci = float(row[5].iloc[0])
        ph = float(row[7].iloc[0][:-1])
        ph_ci = float(row[9].iloc[0])
        amps.append(amp)
        a_cis.append(a_ci)
        phs.append(ph)
        ph_cis.append(ph_ci)
    print(
        f"{amps[0]:.3f}\u00b1{a_cis[0]:.3f}\t{phs[0]:.3f}\u00b1{ph_cis[0]:.3f}\t{amps[1]:.3f}\u00b1{a_cis[1]:.3f}\t{phs[1]:.3f}\u00b1{ph_cis[1]:.3f}\t{amps[2]:.3f}\u00b1{a_cis[2]:.3f}\t{phs[2]:.3f}\u00b1{ph_cis[2]:.3f}"
    )

Constituent	GZ05 2011 Amplitude [m]	GZ05 2011 Phase [°]	GZ05 2013 Amplitude [m]	GZ05 2013 Phase [°]	GZ20 2018 Amplitude [m]	GZ20 2018 Phase [°]	CATS 2011 Amplitude [m]	CATS 2011 Phase [°]
0.126±0.002	219.072±0.846	0.123±0.002	220.045±0.969	0.120±0.002	218.999±1.009
0.139±0.002	164.834±0.691	0.144±0.002	165.279±0.870	0.149±0.002	166.353±0.795
0.112±0.002	130.787±0.908	0.130±0.002	127.133±0.826	0.128±0.002	142.025±0.882
0.047±0.002	156.881±2.059	0.038±0.002	167.066±3.101	0.038±0.002	182.264±3.371
0.452±0.002	189.863±0.196	0.415±0.002	191.888±0.238	0.416±0.002	206.901±0.240
0.368±0.001	193.139±0.233	0.326±0.002	190.866±0.286	0.334±0.002	172.073±0.305
0.143±0.002	194.702±0.607	0.146±0.002	196.141±0.587	0.145±0.002	199.019±0.699
0.078±0.002	181.645±1.128	0.069±0.002	180.984±1.520	0.068±0.002	163.768±1.585
0.047±0.017	178.775±21.283	0.023±0.026	142.586±69.913	0.017±0.016	217.283±64.023
0.011±0.013	196.442±90.717	0.018±0.026	279.761±83.414	0.026±0.019	253.152±42.876
0.017±0.017	212.743±62.643

<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
/var/folders/q8/gfrrq79j2dzfr1jycf0w8bhh0000gx/T/ipykernel_7580/1198683486.py:6: SyntaxWarning: invalid escape sequence '\s'
  dfs = [pd.read_csv(f,header=None, sep='\s+') for f in files]
